# 02 — AraBERT Baseline

**Goal:** fine-tune `aubmindlab/bert-base-arabertv2` on the leakage-free prepared Arabic sentiment splits.

Pipeline:

`prepared train/validation/test → AraBERT preprocessing → real tokenizer length analysis → tokenization → GPU fine-tuning → Accuracy + Macro-F1 → saved baseline model`

The raw dataset is not modified here. AraBERT-specific preprocessing is kept in the model stage so the same path can later be reused by serving.


## 1. Imports and reproducibility

We use:
- PyTorch for GPU training
- AraBERT's official preprocessor
- Hugging Face Transformers for tokenizer/model/training
- Datasets for dataset handling
- scikit-learn for Accuracy, Macro-F1, and the classification report


In [1]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from arabert.preprocess import ArabertPreprocessor
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers import: OK")
print("Datasets import: OK")
print("Seed:", SEED)


e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1+cu132
Transformers import: OK
Datasets import: OK
Seed: 42


## 2. Verify the GPU again

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
print("CUDA build:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
else:
    raise RuntimeError(
        "CUDA is not available. Stop here and fix the PyTorch/CUDA environment before training."
    )


Device: cuda
CUDA available: True
CUDA build: 13.2
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.00 GB


## 3. Project paths and model configuration

The code first looks for prepared split files under `data/processed`.

It accepts common names such as:
- `train.csv`
- `val.csv` / `validation.csv` / `dev.csv`
- `test.csv`

If your `prepare.py` uses different filenames, change only `PROCESSED_DIR` or the candidate names below.


In [3]:
# Find the project root whether Jupyter started from the repo root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_NAME = "aubmindlab/bert-base-arabertv02"

MODEL_DIR = PROJECT_ROOT / "models" / "arabert_baseline"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Model dir    :", MODEL_DIR)
print("Reports dir  :", REPORTS_DIR)

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Prepared-data directory not found: {PROCESSED_DIR}\n"
        "Edit PROCESSED_DIR in this cell to the directory produced by prepare.py."
    )


Project root : e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops
Processed dir: e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\data\processed
Model dir    : e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\models\arabert_baseline
Reports dir  : e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\reports


## 4. Load the prepared train / validation / test splits

In [4]:
SPLIT_CANDIDATES = {
    "train": ["train.csv", "train.parquet"],
    "validation": [
        "validation.csv", "val.csv", "dev.csv",
        "validation.parquet", "val.parquet", "dev.parquet"
    ],
    "test": ["test.csv", "test.parquet"],
}

def find_split(name):
    for filename in SPLIT_CANDIDATES[name]:
        path = PROCESSED_DIR / filename
        if path.exists():
            return path
    return None

split_paths = {name: find_split(name) for name in SPLIT_CANDIDATES}

print("Detected split files:")
for name, path in split_paths.items():
    print(f"  {name:10s}: {path}")

missing = [name for name, path in split_paths.items() if path is None]
if missing:
    print("\nFiles currently visible in data/processed:")
    for p in sorted(PROCESSED_DIR.iterdir()):
        print(" ", p.name)
    raise FileNotFoundError(f"Could not find prepared split(s): {missing}")

def load_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

splits = {name: load_table(path) for name, path in split_paths.items()}

for name, df in splits.items():
    print(f"{name:10s}: shape={df.shape}, columns={list(df.columns)}")


Detected split files:
  train     : e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\data\processed\train.parquet
  validation: e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\data\processed\val.parquet
  test      : e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\data\processed\test.parquet
train     : shape=(70989, 2), columns=['text', 'label']
validation: shape=(8874, 2), columns=['text', 'label']
test      : shape=(8874, 2), columns=['text', 'label']


## 5. Standardize the prepared columns

Expected meaning is:
- `text` = Arabic review text
- `label` = sentiment class

The helper also accepts common alternative column names.


In [5]:
TEXT_CANDIDATES = ["text", "review", "sentence", "comment", "content"]
LABEL_CANDIDATES = ["label", "sentiment", "target", "class"]

def pick_column(df, candidates, kind):
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(
        f"Could not find a {kind} column. Columns found: {list(df.columns)}"
    )

standardized = {}

for split_name, df in splits.items():
    text_col = pick_column(df, TEXT_CANDIDATES, "text")
    label_col = pick_column(df, LABEL_CANDIDATES, "label")

    temp = df[[text_col, label_col]].copy()
    temp.columns = ["text", "label"]
    temp["text"] = temp["text"].astype(str).str.strip()
    standardized[split_name] = temp

train_df = standardized["train"]
val_df = standardized["validation"]
test_df = standardized["test"]

print("Train:")
display(train_df.head())

print("\nSizes:")
print("train:", len(train_df))
print("val  :", len(val_df))
print("test :", len(test_df))


Train:


,text,label
0,ندمت فعلا على قراءة ذا العمل لماركيز لا أعرف إ...,negative
1,مخيب للأمل كثرة البعوض في المكان وكثرة الاصوات...,negative
2,افضل فندق سكنت في دبي لمدة سنوات كل شي كما أن ...,positive
3,مرضي الموقع سل الوصول للحرمالقيمة المالية عدم ...,negative
4,ضعيف النظافة جدا جدا سيئة,negative



Sizes:
train: 70989
val  : 8874
test : 8874


## 6. Normalize sentiment labels

The prepared corpus uses the three sentiment classes:

`NEGATIVE`, `NEUTRAL`, `POSITIVE`

If `prepare.py` already writes numeric labels, the existing dataset mapping is interpreted as:

`0 = NEGATIVE`, `1 = NEUTRAL`, `2 = POSITIVE`.


In [6]:
LABEL2ID = {
    "NEGATIVE": 0,
    "NEUTRAL": 1,
    "POSITIVE": 2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def normalize_label(value):
    if pd.isna(value):
        raise ValueError("Missing label found in prepared data.")

    if isinstance(value, str):
        s = value.strip().upper()
        if s in LABEL2ID:
            return LABEL2ID[s]
        if s.isdigit() and int(s) in ID2LABEL:
            return int(s)

    if isinstance(value, (int, np.integer)) and int(value) in ID2LABEL:
        return int(value)

    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        ivalue = int(value)
        if ivalue in ID2LABEL:
            return ivalue

    raise ValueError(f"Unknown label value: {value!r}")

for name, df in standardized.items():
    df["labels"] = df["label"].map(normalize_label)

    if df["labels"].isna().any():
        raise ValueError(f"Missing normalized labels in {name}")

    print(f"{name:10s}:")
    print(df["labels"].value_counts().sort_index().rename(index=ID2LABEL))

print("\nLabel mapping:", ID2LABEL)


train     :
labels
NEGATIVE    33107
NEUTRAL     20215
POSITIVE    17667
Name: count, dtype: int64
validation:
labels
NEGATIVE    4138
NEUTRAL     2527
POSITIVE    2209
Name: count, dtype: int64
test      :
labels
NEGATIVE    4139
NEUTRAL     2527
POSITIVE    2208
Name: count, dtype: int64

Label mapping: {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}


## 7. Leakage sanity check

The split boundaries were created by `prepare.py`. This notebook does not rebuild them.

We verify the important property before training: **no exact text overlap between train, validation, and test**.


In [7]:
def overlap(a, b):
    return len(set(a["text"]) & set(b["text"]))

print("train ∩ val :", overlap(train_df, val_df))
print("train ∩ test:", overlap(train_df, test_df))
print("val   ∩ test:", overlap(val_df, test_df))

assert overlap(train_df, val_df) == 0
assert overlap(train_df, test_df) == 0
assert overlap(val_df, test_df) == 0

print("Leakage check: PASSED")


train ∩ val : 0
train ∩ test: 0
val   ∩ test: 0
Leakage check: PASSED


## 8. Load the official AraBERT v2 preprocessor and tokenizer

AraBERT's official repository recommends applying its preprocessing before training/testing. For `bert-base-arabertv2`, the preprocessor automatically enables the model's required Farasa segmentation.

This is the beginning of the **model-specific** text path; it is not additional generic dataset cleaning.


In [8]:
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)


e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BSM Computer\.cache\huggingface\hub\models--aubmindlab--bert-base-arabertv02. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model: aubmindlab/bert-base-arabertv02
Tokenizer: BertTokenizer
Vocab size: 64000
Model max length: 512


## 9. Verify preprocessing on real Arabic examples

In [9]:
examples = train_df["text"].head(5).tolist()

for i, text in enumerate(examples, start=1):
    processed = arabert_prep.preprocess(text)
    print(f"Example {i}")
    print("Original:", text)
    print("AraBERT :", processed)
    print("-" * 100)


Example 1
Original: ندمت فعلا على قراءة ذا العمل لماركيز لا أعرف إن كانت المشكلة من العمل أم من الترجمة
AraBERT : ندمت فعلا على قراءة ذا العمل لماركيز لا أعرف إن كانت المشكلة من العمل أم من الترجمة
----------------------------------------------------------------------------------------------------
Example 2
Original: مخيب للأمل كثرة البعوض في المكان وكثرة الاصوات المزعج
AraBERT : مخيب للأمل كثرة البعوض في المكان وكثرة الاصوات المزعج
----------------------------------------------------------------------------------------------------
Example 3
Original: افضل فندق سكنت في دبي لمدة سنوات كل شي كما أن يسمح بادخال الضيوف لا يوجد شيء
AraBERT : افضل فندق سكنت في دبي لمدة سنوات كل شي كما أن يسمح بادخال الضيوف لا يوجد شيء
----------------------------------------------------------------------------------------------------
Example 4
Original: مرضي الموقع سل الوصول للحرمالقيمة المالية عدم وجود بالغرف اللوبي فقط
AraBERT : مرضي الموقع سل الوصول للحرمالقيمة المالية عدم وجود بالغرف اللوبي فقط
---------

## 10. Apply AraBERT preprocessing

Keep the original prepared `text` untouched and create:

`text_arabert` = text after the exact preprocessing required by the selected AraBERT checkpoint.


In [10]:
def preprocess_dataframe(df, split_name):
    result = df.copy()
    tqdm.pandas(desc=f"AraBERT preprocessing: {split_name}")
    result["text_arabert"] = result["text"].progress_apply(arabert_prep.preprocess)
    return result

train_df = preprocess_dataframe(train_df, "train")
val_df = preprocess_dataframe(val_df, "validation")
test_df = preprocess_dataframe(test_df, "test")

display(train_df[["text", "text_arabert"]].head(3))


AraBERT preprocessing: test: 100%|██████████| 8874/8874 [00:00<00:00, 9055.64it/s]


,text,text_arabert
0,ندمت فعلا على قراءة ذا العمل لماركيز لا أعرف إ...,ندمت فعلا على قراءة ذا العمل لماركيز لا أعرف إ...
1,مخيب للأمل كثرة البعوض في المكان وكثرة الاصوات...,مخيب للأمل كثرة البعوض في المكان وكثرة الاصوات...
2,افضل فندق سكنت في دبي لمدة سنوات كل شي كما أن ...,افضل فندق سكنت في دبي لمدة سنوات كل شي كما أن ...


## 11. Measure REAL AraBERT token lengths

Character/word counts from the data-understanding notebook are descriptive only. Here we measure how many tokens AraBERT actually receives after its model-specific preprocessing.


In [11]:
all_model_texts = pd.concat(
    [
        train_df[["text_arabert"]],
        val_df[["text_arabert"]],
        test_df[["text_arabert"]],
    ],
    ignore_index=True,
)["text_arabert"].tolist()

def measure_token_lengths(texts, batch_size=256):
    lengths = []
    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Token length analysis"
    ):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])
    return np.asarray(lengths, dtype=np.int32)

token_lengths = measure_token_lengths(all_model_texts)

summary = pd.Series(token_lengths).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)
display(summary)

print("\nShare exceeding candidate lengths:")
for limit in [64, 128, 256, 384, 512]:
    share = (token_lengths > limit).mean()
    print(f"  > {limit:3d}: {share:.2%}")


Token length analysis: 100%|██████████| 347/347 [00:02<00:00, 136.87it/s]


count    88737.000000
mean        37.716578
std         80.456683
min          2.000000
50%         18.000000
75%         30.000000
90%         74.000000
95%        132.000000
99%        361.000000
max       1908.000000
dtype: float64


Share exceeding candidate lengths:
  >  64: 11.60%
  > 128: 5.21%
  > 256: 1.93%
  > 384: 0.88%
  > 512: 0.51%


## 12. Choose `max_length`

We use the observed 95th percentile, round upward to a multiple of 32, and cap it at the model's 512-token limit.

The printed value is the value that training will actually use.


In [12]:
p95 = int(np.percentile(token_lengths, 95))

MAX_LENGTH = int(min(512, math.ceil(p95 / 32) * 32))
MAX_LENGTH = max(32, MAX_LENGTH)

print("P95 token length :", p95)
print("Chosen max_length:", MAX_LENGTH)

truncation_rate = (token_lengths > MAX_LENGTH).mean()
print(
    f"Estimated truncation rate at max_length={MAX_LENGTH}: "
    f"{truncation_rate:.2%}"
)


P95 token length : 132
Chosen max_length: 160
Estimated truncation rate at max_length=160: 3.88%


## 13. Build Hugging Face datasets and tokenize

In [13]:
def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[["text_arabert", "labels"]],
        preserve_index=False,
    )

raw_datasets = DatasetDict({
    "train": to_hf_dataset(train_df),
    "validation": to_hf_dataset(val_df),
    "test": to_hf_dataset(test_df),
})

def tokenize_batch(batch):
    encoded = tokenizer(
        batch["text_arabert"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    encoded["labels"] = batch["labels"]
    return encoded

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text_arabert"],
    desc="Tokenizing",
)

print(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())


Tokenizing: 100%|██████████| 8874/8874 [00:00<00:00, 35740.08 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 70989
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8874
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8874
    })
})
dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


## 14. Dynamic padding collator

In [14]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

print("Data collator: OK")


Data collator: OK


## 15. Load the AraBERT sequence-classification model

The pretrained AraBERT encoder gets a 3-class classification head for:

`NEGATIVE`, `NEUTRAL`, `POSITIVE`.


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

print("Model loaded.")
print("Parameters:", sum(p.numel() for p in model.parameters()))


e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BSM Computer\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 822.49it/s]
[transformers] Ber

Model loaded.
Parameters: 135195651


## 16. Metrics: Accuracy + Macro-F1

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

print("Metric function: OK")


Metric function: OK


## 17. Training configuration

This is a first reproducible baseline, not a hyperparameter sweep.

Settings are intentionally conservative for an RTX 3060 Laptop GPU:
- AraBERT base
- batch size 4
- gradient accumulation 4
- FP16 on CUDA
- gradient checkpointing
- 2 epochs
- best checkpoint selected by validation Macro-F1


In [18]:
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=0.1,  # ✅ replaces warmup_ratio=0.1

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    dataloader_num_workers=0,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

print(training_args)


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=F

## 18. Create the Trainer

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer: OK")


Trainer: OK


## 19. Train the AraBERT baseline

**This is the long-running cell.**

Training uses the leakage-free prepared train split and selects the best checkpoint using validation Macro-F1.


In [20]:
train_result = trainer.train()

print("Training complete.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.378575,0.336042,0.860379,0.854380
2,0.962783,0.344909,0.873112,0.866439


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.88it/s]


Training complete.
Best checkpoint: e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\models\arabert_baseline\checkpoints\checkpoint-8874


## 20. Evaluate on the untouched test split

In [21]:
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["test"],
)

print("Test metrics:")
for key, value in test_metrics.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.962783,0.348608,2,0.873000,0.866321


Test metrics:
eval_loss: 0.3486
eval_accuracy: 0.8730
eval_f1_macro: 0.8663


## 21. Classification report and confusion matrix

In [22]:
pred_output = trainer.predict(tokenized_datasets["test"])
test_predictions = np.argmax(pred_output.predictions, axis=-1)
test_labels = pred_output.label_ids

print(
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1, 2],
        target_names=[ID2LABEL[i] for i in [0, 1, 2]],
        digits=4,
    )
)

cm = confusion_matrix(test_labels, test_predictions, labels=[0, 1, 2])
cm_df = pd.DataFrame(
    cm,
    index=[ID2LABEL[i] for i in [0, 1, 2]],
    columns=[ID2LABEL[i] for i in [0, 1, 2]],
)

display(cm_df)


              precision    recall  f1-score   support

    NEGATIVE     0.9232    0.8884    0.9054      4139
     NEUTRAL     0.8145    0.8480    0.8309      2527
    POSITIVE     0.8527    0.8727    0.8626      2208

    accuracy                         0.8730      8874
   macro avg     0.8634    0.8697    0.8663      8874
weighted avg     0.8747    0.8730    0.8736      8874



,NEGATIVE,NEUTRAL,POSITIVE
NEGATIVE,3677,325,137
NEUTRAL,188,2143,196
POSITIVE,118,163,1927


## 22. Save the baseline model, tokenizer, and reproducibility metadata

In [23]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

metadata = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "tokenizer_model_max_length": int(tokenizer.model_max_length),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "p95_token_length": p95,
    "estimated_truncation_rate": float(truncation_rate),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "test_metrics": {
        k: float(v) for k, v in test_metrics.items()
        if isinstance(v, (float, int, np.floating, np.integer))
    },
    "torch_version": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
}

with open(MODEL_DIR / "baseline_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with open(REPORTS_DIR / "arabert_baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved model/tokenizer to:", MODEL_DIR)
print("Saved metrics to:", REPORTS_DIR / "arabert_baseline_metrics.json")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.55it/s]

Saved model/tokenizer to: e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\models\arabert_baseline
Saved metrics to: e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\reports\arabert_baseline_metrics.json


## 23. Quick inference smoke test

In [24]:
def predict_sentiment(text: str):
    processed = arabert_prep.preprocess(text)

    inputs = tokenizer(
        processed,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}

    trainer.model.eval()
    with torch.inference_mode():
        logits = trainer.model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    pred_id = int(torch.argmax(probabilities).item())

    return {
        "label": ID2LABEL[pred_id],
        "confidence": float(probabilities[pred_id].item()),
        "model_version": MODEL_NAME,
    }

predict_sentiment("الخدمة ممتازة والتجربة رائعة")


{'label': 'POSITIVE',
 'confidence': 0.9934237003326416,
 'model_version': 'aubmindlab/bert-base-arabertv02'}

## 24. Final baseline conclusion

Record the actual results after execution:

- Model: `aubmindlab/bert-base-arabertv2`
- Prepared split sizes: shown above
- Real tokenizer P95: shown above
- Chosen `max_length`: shown above
- Test Accuracy: shown above
- Test Macro-F1: shown above
- Saved model: `models/arabert_baseline/`
- Metrics: `reports/arabert_baseline_metrics.json`

This notebook establishes the first AraBERT baseline. MLflow tracking, production serving, BentoML, ONNX/TensorRT, benchmarking, canary deployment, and monitoring belong to the later MLOps stages.


In [26]:
# ============================================================
# AraBERTv0.2 Hyperparameter Tuning + MLflow Tracking
# ============================================================

from pathlib import Path
import gc
import json
import shutil
import subprocess
import time

import mlflow
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


# ------------------------------------------------------------
# 1. Fixed experiment settings
# ------------------------------------------------------------

SEED = 42
MODEL_NAME = "aubmindlab/bert-base-arabertv02"

set_seed(SEED)

assert torch.cuda.is_available(), (
    "CUDA is not available. Fix the GPU environment before running the sweep."
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

TUNING_DIR = PROJECT_ROOT / "models" / "arabert_v02_tuning"
BEST_MODEL_DIR = PROJECT_ROOT / "models" / "arabert_v02_best"
TUNING_REPORTS_DIR = REPORTS_DIR / "arabert_v02_tuning"

TUNING_DIR.mkdir(parents=True, exist_ok=True)
TUNING_REPORTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. MLflow setup
# ------------------------------------------------------------
#
# We use MLflow Tracking here.
#
# Every experiment gets its own MLflow run.
# We log parameters + validation metrics + evaluation reports.
#
# We intentionally do NOT register every trial as a model version.
# That would create many large model copies and is unnecessary
# during exploration.
#
# Model Registry comes after we select the final candidate.
# ------------------------------------------------------------

MLFLOW_DB = PROJECT_ROOT / "mlflow.db"

# Database-backed MLflow tracking.
# SQLite is appropriate for local development.
MLFLOW_URI = f"sqlite:///{MLFLOW_DB.as_posix()}"

mlflow.set_tracking_uri(MLFLOW_URI)

EXPERIMENT_NAME = "arabic-sentiment-arabert-v02-tuning"

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow version:", mlflow.__version__)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiment:", EXPERIMENT_NAME)
print("MLflow database:", MLFLOW_DB)

# ------------------------------------------------------------
# 4. Git commit for reproducibility
# ------------------------------------------------------------

try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except Exception:
    git_commit = "unknown"

print("Git commit:", git_commit)


# ------------------------------------------------------------
# 5. Load tokenizer once
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)


# ------------------------------------------------------------
# 6. Define experiment configurations
# ------------------------------------------------------------
#
# Baseline:
#   lr = 2e-5
#   epochs = 2
#   max_length = 160
#
# We keep:
#   weight_decay = 0.01
#   warmup_steps = 0.1
#   effective batch size = 16
#
# fixed, so that we can understand the effect of each change.
# ------------------------------------------------------------

TRIALS = [
    {
        "name": "lr_1e-5",
        "learning_rate": 1e-5,
        "num_train_epochs": 2,
        "max_length": 160,
    },
    {
        "name": "lr_3e-5",
        "learning_rate": 3e-5,
        "num_train_epochs": 2,
        "max_length": 160,
    },
    {
        "name": "epochs_3",
        "learning_rate": 2e-5,
        "num_train_epochs": 3,
        "max_length": 160,
    },
    {
        "name": "max_length_256",
        "learning_rate": 2e-5,
        "num_train_epochs": 2,
        "max_length": 256,
    },
]


# ------------------------------------------------------------
# 7. Reuse tokenized datasets when max_length is the same
# ------------------------------------------------------------

tokenized_cache = {}


def build_tokenized_datasets(max_length: int):
    """Build tokenized HF datasets for one max_length value."""

    if max_length in tokenized_cache:
        return tokenized_cache[max_length]

    print(f"\nTokenizing datasets with max_length={max_length} ...")

    def to_hf_dataset(df: pd.DataFrame) -> Dataset:
        return Dataset.from_pandas(
            df[["text", "labels"]],
            preserve_index=False,
        )

    raw_datasets = DatasetDict(
        {
            "train": to_hf_dataset(train_df),
            "validation": to_hf_dataset(val_df),
            "test": to_hf_dataset(test_df),
        }
    )

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
        )

    tokenized = raw_datasets.map(
        tokenize_batch,
        batched=True,
        desc=f"Tokenizing (max_length={max_length})",
    )

    tokenized_cache[max_length] = tokenized

    print("Tokenization complete.")

    return tokenized


# ------------------------------------------------------------
# 8. Metric helpers
# ------------------------------------------------------------

def validation_metrics(trainer: Trainer, dataset):
    """Evaluate on validation data."""
    metrics = trainer.evaluate(dataset)

    return {
        "accuracy": float(metrics["eval_accuracy"]),
        "f1_macro": float(metrics["eval_f1_macro"]),
        "loss": float(metrics["eval_loss"]),
    }


def save_evaluation_report(
    trainer: Trainer,
    dataset,
    output_dir: Path,
):
    """Create a detailed validation report."""

    prediction_output = trainer.predict(dataset)

    predictions = np.argmax(
        prediction_output.predictions,
        axis=-1,
    )

    labels = prediction_output.label_ids

    report = classification_report(
        labels,
        predictions,
        labels=[0, 1, 2],
        target_names=[ID2LABEL[i] for i in [0, 1, 2]],
        output_dict=True,
        digits=4,
    )

    cm = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1, 2],
    )

    report_json = output_dir / "validation_classification_report.json"

    with open(report_json, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    cm_df = pd.DataFrame(
        cm,
        index=[ID2LABEL[i] for i in [0, 1, 2]],
        columns=[ID2LABEL[i] for i in [0, 1, 2]],
    )

    cm_csv = output_dir / "validation_confusion_matrix.csv"
    cm_df.to_csv(cm_csv)

    return report, cm_df


# ------------------------------------------------------------
# 9. Reference baseline
# ------------------------------------------------------------
#
# IMPORTANT:
# We do NOT retrain the baseline.
#
# We already trained it.
# We keep that historical model untouched in:
#
# models/arabert_baseline/
#
# We simply include its recorded result in the comparison.
#
# Validation Macro-F1 from the completed baseline run:
# 0.866439
#
# Test Macro-F1:
# 0.8663
# ------------------------------------------------------------

results = []

baseline_result = {
    "run": "baseline_existing",
    "learning_rate": 2e-5,
    "num_train_epochs": 2,
    "max_length": 160,
    "weight_decay": 0.01,
    "warmup_steps": 0.1,
    "val_accuracy": 0.873112,
    "val_f1_macro": 0.866439,
    "val_loss": 0.344909,
    "test_accuracy": 0.8730,
    "test_f1_macro": 0.8663,
    "model_path": str(PROJECT_ROOT / "models" / "arabert_baseline"),
}

results.append(baseline_result)

# Track the already-completed baseline in MLflow.
with mlflow.start_run(run_name="baseline_existing"):

    mlflow.set_tags(
        {
            "model_family": "AraBERTv0.2",
            "model_name": MODEL_NAME,
            "stage": "baseline_reference",
            "git_commit": git_commit,
        }
    )

    mlflow.log_params(
        {
            "model_name": MODEL_NAME,
            "learning_rate": 2e-5,
            "num_train_epochs": 2,
            "max_length": 160,
            "weight_decay": 0.01,
            "warmup_steps": 0.1,
            "train_batch_size": 4,
            "eval_batch_size": 8,
            "gradient_accumulation_steps": 4,
            "effective_batch_size": 16,
            "seed": SEED,
            "fp16": True,
            "gradient_checkpointing": True,
        }
    )

    mlflow.log_metrics(
        {
            "val_accuracy": 0.873112,
            "val_f1_macro": 0.866439,
            "val_loss": 0.344909,
            "test_accuracy": 0.8730,
            "test_f1_macro": 0.8663,
        }
    )

    baseline_metadata = TUNING_REPORTS_DIR / "baseline_reference.json"

    with open(baseline_metadata, "w", encoding="utf-8") as f:
        json.dump(
            baseline_result,
            f,
            ensure_ascii=False,
            indent=2,
        )

    mlflow.log_artifact(baseline_metadata)


# ------------------------------------------------------------
# 10. Initialize selection
# ------------------------------------------------------------

best_val_f1 = baseline_result["val_f1_macro"]
best_run = baseline_result["run"]
selected_model_dir = PROJECT_ROOT / "models" / "arabert_baseline"

print("\n" + "=" * 70)
print("BASELINE REFERENCE")
print("=" * 70)
print(f"Validation Accuracy : {baseline_result['val_accuracy']:.4f}")
print(f"Validation Macro-F1 : {baseline_result['val_f1_macro']:.4f}")
print(f"Test Accuracy       : {baseline_result['test_accuracy']:.4f}")
print(f"Test Macro-F1       : {baseline_result['test_f1_macro']:.4f}")

PyTorch: 2.14.1+cu132
CUDA: 13.2
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


2026/10/01 20:56:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/01 20:56:59 INFO mlflow.store.db.utils: Updating database tables
2026/10/01 20:57:00 INFO mlflow.tracking.fluent: Experiment with name 'arabic-sentiment-arabert-v02-tuning' does not exist. Creating a new experiment.


MLflow version: 3.16.1
MLflow tracking URI: sqlite:///e:/03 Education/Online Courses/MLOps MENA/Final Project/arabic-sentiment-mlops/mlflow.db
MLflow experiment: arabic-sentiment-arabert-v02-tuning
MLflow database: e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\mlflow.db
Git commit: 7239ac6441443ba09c9ada817038f034dac656d9
Tokenizer: BertTokenizer
Vocabulary size: 64000
Model max length: 512

BASELINE REFERENCE
Validation Accuracy : 0.8731
Validation Macro-F1 : 0.8664
Test Accuracy       : 0.8730
Test Macro-F1       : 0.8663


In [27]:
# ------------------------------------------------------------
# 11. Run all tuning experiments
# ------------------------------------------------------------

for trial_number, config in enumerate(TRIALS, start=1):

    trial_name = f"trial_{trial_number:02d}_{config['name']}"
    trial_dir = TUNING_DIR / trial_name
    report_dir = TUNING_REPORTS_DIR / trial_name

    trial_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"STARTING {trial_name}")
    print("=" * 70)
    print(config)

    set_seed(SEED)

    # Build/reuse tokenized data for this max_length.
    tokenized_datasets_trial = build_tokenized_datasets(
        config["max_length"]
    )

    # Fresh model for every experiment.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        label2id=LABEL2ID,
        id2label=ID2LABEL,
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8,
    )

    training_args = TrainingArguments(
        output_dir=str(trial_dir / "checkpoints"),

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=100,

        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,

        num_train_epochs=config["num_train_epochs"],
        learning_rate=config["learning_rate"],
        weight_decay=0.01,

        # Transformers 5.x:
        # float < 1 means ratio of total training steps.
        warmup_steps=0.1,

        fp16=True,
        gradient_checkpointing=True,

        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,

        dataloader_num_workers=0,
        report_to="none",

        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets_trial["train"],
        eval_dataset=tokenized_datasets_trial["validation"],
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    start_time = time.perf_counter()

    with mlflow.start_run(run_name=trial_name):

        # --------------------------------------------
        # MLflow tags
        # --------------------------------------------

        mlflow.set_tags(
            {
                "model_family": "AraBERTv0.2",
                "model_name": MODEL_NAME,
                "stage": "hyperparameter_tuning",
                "trial": trial_name,
                "git_commit": git_commit,
            }
        )

        # --------------------------------------------
        # MLflow parameters
        # --------------------------------------------

        mlflow.log_params(
            {
                "model_name": MODEL_NAME,
                "learning_rate": config["learning_rate"],
                "num_train_epochs": config["num_train_epochs"],
                "max_length": config["max_length"],
                "weight_decay": 0.01,
                "warmup_steps": 0.1,
                "train_batch_size": 4,
                "eval_batch_size": 8,
                "gradient_accumulation_steps": 4,
                "effective_batch_size": 16,
                "seed": SEED,
                "fp16": True,
                "gradient_checkpointing": True,
            }
        )

        # --------------------------------------------
        # Train
        # --------------------------------------------

        train_result = trainer.train()

        elapsed_seconds = time.perf_counter() - start_time

        # --------------------------------------------
        # Validation evaluation
        # --------------------------------------------

        val_result = validation_metrics(
            trainer,
            tokenized_datasets_trial["validation"],
        )

        # Save validation report.
        report, cm_df = save_evaluation_report(
            trainer,
            tokenized_datasets_trial["validation"],
            report_dir,
        )

        # --------------------------------------------
        # Log metrics
        # --------------------------------------------

        mlflow.log_metrics(
            {
                "val_accuracy": val_result["accuracy"],
                "val_f1_macro": val_result["f1_macro"],
                "val_loss": val_result["loss"],
                "training_time_seconds": float(elapsed_seconds),
            }
        )

        # Log per-class validation F1.
        for label_name in ["NEGATIVE", "NEUTRAL", "POSITIVE"]:
            class_f1 = report[label_name]["f1-score"]

            mlflow.log_metric(
                f"val_f1_{label_name.lower()}",
                float(class_f1),
            )

        # Log small evaluation artifacts.
        mlflow.log_artifacts(
            str(report_dir),
            artifact_path="evaluation",
        )

        # --------------------------------------------
        # Store result
        # --------------------------------------------

        current_result = {
            "run": trial_name,
            "learning_rate": config["learning_rate"],
            "num_train_epochs": config["num_train_epochs"],
            "max_length": config["max_length"],
            "weight_decay": 0.01,
            "warmup_steps": 0.1,
            "val_accuracy": val_result["accuracy"],
            "val_f1_macro": val_result["f1_macro"],
            "val_loss": val_result["loss"],
            "training_time_minutes": elapsed_seconds / 60.0,
            "model_path": str(trial_dir),
        }

        results.append(current_result)

        # --------------------------------------------
        # Print immediate result
        # --------------------------------------------

        print(
            f"\n{trial_name} completed"
            f"\nValidation Accuracy : {val_result['accuracy']:.4f}"
            f"\nValidation Macro-F1 : {val_result['f1_macro']:.4f}"
            f"\nValidation Loss     : {val_result['loss']:.4f}"
            f"\nTraining Time       : {elapsed_seconds / 60.0:.1f} min"
        )

        # --------------------------------------------
        # Update best model
        # --------------------------------------------
        #
        # Selection criterion = VALIDATION Macro-F1.
        #
        # Test is not used here.
        # --------------------------------------------

        if val_result["f1_macro"] > best_val_f1:

            best_val_f1 = val_result["f1_macro"]
            best_run = trial_name

            # Replace the "best so far" model only.
            # Other experiment history remains in MLflow.
            if BEST_MODEL_DIR.exists():
                shutil.rmtree(BEST_MODEL_DIR)

            trainer.save_model(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)

            selected_model_dir = BEST_MODEL_DIR

            best_summary = {
                "selected_run": trial_name,
                "selection_metric": "validation_f1_macro",
                "validation_f1_macro": best_val_f1,
                "configuration": config,
            }

            with open(
                REPORTS_DIR / "arabert_v02_best_summary.json",
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    best_summary,
                    f,
                    ensure_ascii=False,
                    indent=2,
                )

            print("\n*** NEW BEST VALIDATION MODEL ***")
            print("Run:", best_run)
            print(f"Validation Macro-F1: {best_val_f1:.4f}")

    # --------------------------------------------
    # Free GPU / CPU memory
    # --------------------------------------------

    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()

    # --------------------------------------------
    # Delete trial checkpoint directory.
    #
    # MLflow keeps the experiment history.
    # We only keep the current global best model locally.
    # --------------------------------------------

    if trial_dir.exists():
        shutil.rmtree(trial_dir)


# ------------------------------------------------------------
# 12. Display complete tuning results
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "val_f1_macro",
    ascending=False,
).reset_index(drop=True)

print("\n" + "=" * 70)
print("ALL EXPERIMENT RESULTS")
print("=" * 70)

display(
    results_df[
        [
            "run",
            "learning_rate",
            "num_train_epochs",
            "max_length",
            "val_accuracy",
            "val_f1_macro",
            "val_loss",
            "training_time_minutes",
        ]
    ]
)


# ------------------------------------------------------------
# 13. Final selected configuration
# ------------------------------------------------------------

selected_row = results_df.iloc[0]

print("\n" + "=" * 70)
print("SELECTED CONFIGURATION")
print("=" * 70)

print("Run:", selected_row["run"])
print("Learning rate:", selected_row["learning_rate"])
print("Epochs:", selected_row["num_train_epochs"])
print("Max length:", selected_row["max_length"])
print(f"Validation Accuracy: {selected_row['val_accuracy']:.4f}")
print(f"Validation Macro-F1: {selected_row['val_f1_macro']:.4f}")
print("Model path:", selected_model_dir)


# ------------------------------------------------------------
# 14. FINAL TEST EVALUATION
# ------------------------------------------------------------
#
# This is the ONLY place where we use the test split to decide
# the final reported test performance.
# ------------------------------------------------------------

final_model = AutoModelForSequenceClassification.from_pretrained(
    selected_model_dir,
)

final_tokenizer = AutoTokenizer.from_pretrained(
    selected_model_dir,
)

final_max_length = int(selected_row["max_length"])

final_tokenized = Dataset.from_pandas(
    test_df[["text", "labels"]],
    preserve_index=False,
)


def final_tokenize(batch):
    return final_tokenizer(
        batch["text"],
        truncation=True,
        max_length=final_max_length,
    )


final_test_dataset = final_tokenized.map(
    final_tokenize,
    batched=True,
    desc="Tokenizing final test set",
)

final_collator = DataCollatorWithPadding(
    tokenizer=final_tokenizer,
    pad_to_multiple_of=8,
)

final_args = TrainingArguments(
    output_dir=str(TUNING_REPORTS_DIR / "final_test_eval"),
    per_device_eval_batch_size=8,
    report_to="none",
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    data_collator=final_collator,
    processing_class=final_tokenizer,
    compute_metrics=compute_metrics,
)

final_test_metrics = final_trainer.evaluate(
    final_test_dataset,
)

final_predictions_output = final_trainer.predict(
    final_test_dataset,
)

final_predictions = np.argmax(
    final_predictions_output.predictions,
    axis=-1,
)

final_labels = final_predictions_output.label_ids

final_report = classification_report(
    final_labels,
    final_predictions,
    labels=[0, 1, 2],
    target_names=[ID2LABEL[i] for i in [0, 1, 2]],
    digits=4,
)

final_cm = confusion_matrix(
    final_labels,
    final_predictions,
    labels=[0, 1, 2],
)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Test Accuracy: {final_test_metrics['eval_accuracy']:.4f}")
print(f"Test Macro-F1: {final_test_metrics['eval_f1_macro']:.4f}")
print(f"Test Loss: {final_test_metrics['eval_loss']:.4f}")

print("\nClassification report:")
print(final_report)

final_cm_df = pd.DataFrame(
    final_cm,
    index=[ID2LABEL[i] for i in [0, 1, 2]],
    columns=[ID2LABEL[i] for i in [0, 1, 2]],
)

print("Confusion matrix:")
display(final_cm_df)


# ------------------------------------------------------------
# 15. Save final tuning summary
# ------------------------------------------------------------

final_summary = {
    "model_name": MODEL_NAME,
    "selected_run": str(selected_row["run"]),
    "selection_metric": "validation_f1_macro",
    "selected_validation_f1_macro": float(
        selected_row["val_f1_macro"]
    ),
    "selected_configuration": {
        "learning_rate": float(selected_row["learning_rate"]),
        "num_train_epochs": int(selected_row["num_train_epochs"]),
        "max_length": int(selected_row["max_length"]),
        "weight_decay": 0.01,
        "warmup_steps": 0.1,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "effective_batch_size": 16,
        "fp16": True,
        "gradient_checkpointing": True,
        "seed": SEED,
    },
    "final_test_metrics": {
        "accuracy": float(final_test_metrics["eval_accuracy"]),
        "f1_macro": float(final_test_metrics["eval_f1_macro"]),
        "loss": float(final_test_metrics["eval_loss"]),
    },
    "model_path": str(selected_model_dir),
    "git_commit": git_commit,
}

with open(
    REPORTS_DIR / "arabert_v02_tuning_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

# Small MLflow summary artifact.
summary_path = TUNING_REPORTS_DIR / "final_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        final_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved final tuning summary to:")
print(REPORTS_DIR / "arabert_v02_tuning_summary.json")

print("\nMLflow database:")
print(MLFLOW_DB)

print("\nDONE.")


STARTING trial_01_lr_1e-5
{'name': 'lr_1e-5', 'learning_rate': 1e-05, 'num_train_epochs': 2, 'max_length': 160}

Tokenizing datasets with max_length=160 ...


Tokenizing (max_length=160): 100%|██████████| 8874/8874 [00:00<00:00, 13929.20 examples/s]


Tokenization complete.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 488.06it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.405241,0.353153,0.853392,0.846086
2,1.115122,0.351227,0.865450,0.857881


...

Tokenizing final test set: 100%|██████████| 8874/8874 [00:00<00:00, 27514.65 examples/s]


Training Loss,Validation Loss,Step,Accuracy,F1 Macro
No log,0.438872,0,0.875704,0.868113



FINAL TEST RESULTS
Test Accuracy: 0.8757
Test Macro-F1: 0.8681
Test Loss: 0.4389

Classification report:
              precision    recall  f1-score   support

    NEGATIVE     0.9223    0.9007    0.9114      4139
     NEUTRAL     0.8163    0.8421    0.8290      2527
    POSITIVE     0.8607    0.8673    0.8640      2208

    accuracy                         0.8757      8874
   macro avg     0.8664    0.8700    0.8681      8874
weighted avg     0.8768    0.8757    0.8761      8874

Confusion matrix:


,NEGATIVE,NEUTRAL,POSITIVE
NEGATIVE,3728,292,119
NEUTRAL,208,2128,191
POSITIVE,106,187,1915



Saved final tuning summary to:
e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\reports\arabert_v02_tuning_summary.json

MLflow database:
e:\03 Education\Online Courses\MLOps MENA\Final Project\arabic-sentiment-mlops\mlflow.db

DONE.
